🔰PyTorchでニューラルネットワーク基礎　#43 【llama化・KVキャッシュ】


### 内容
* Qiitaの記事と連動しています
* 各種ファイルの保存先は環境によって適宜変更してください

### データについて
* Livedoorニュースコーパスのlivedoor-hommeカテゴリを利用します。
* huggingfaceの　llm-book/livedoor-news-corpus　などから適宜ダウンロードしてください。
* 先頭から3行のURL、日付、タイトルを削除したものを利用します。

### トークナイザーについて
* tokenizer/livedoor_tokenizer_10k.json
    * bytelevel BPEで構成した語彙数10kのtokenizer


### 今回扱う内容
* LayerNormをRMSNormに変更
* MLP層の活性化関数をSwiGLUに変更
* dynamic KVキャッシュの実装
* 絶対位置埋め込みを利用（RoPEではない）

### 注意点
* 汎用的な内容は生成不能
* 学習した内容、学習した文と類似文章が入力されるとうまく生成される
* 学習していない内容は、全くだめ
* モデルサイズ・データサイズが小さいので完全コピーに近い生成がみられるがこれが正常な状況
* 系列長を256とある程度長くしないと、指示チューニング時にトークン不足となる
* 事前学習だけを考える場合、GPUメモリとの兼ね合いで seq_len=128でも事前学習による生成可能

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer
from pathlib import Path


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"{device=}")

In [ ]:
data_dir = "./data/homme_texts/"                             # 学習用データのディレクトリ
tokenizer_filename = "tokenizer/livedoor_tokenizer_10k.json"   # tokenizer
pretrain_filename = "model/kv_homme_seq_256_bpe_10k.model"


# 指定したファイル・ディレクトリのチェック
# 先にチェック
for item in [data_dir, tokenizer_filename]:
    assert Path(item).exists(), f"必要なファイル/ディレクトリが見つかりません: {item}"

# トークナイザー
tokenizer = Tokenizer.from_file(tokenizer_filename)
print(f"size: {tokenizer.get_vocab_size()}")

In [ ]:
class ModelConfig:
    def __init__(self, tokenizer):
        # モデル構造
        self.vocab_size = tokenizer.get_vocab_size()
        self.seq_len = 256
        self.d_model = 256
        self.nhead = 8
        self.dim_feedforward = 4*self.d_model
        self.num_layers = 6
        self.dropout = 0.1

        # 特殊トークンID
        self.pad_token_id = tokenizer.token_to_id("<pad>")
        self.eod_token_id = tokenizer.token_to_id("<eod>")

        # 学習データに関する設定
        self.context_size = self.seq_len         # 学習できる長さ

        # 学習設定
        self.batch_size = 256
        self.learning_rate = 0.001  # これだとデフォルトと変わらない
        self.num_steps = 2500
        self.max_grad_norm = 1.0
        self.ignore_index = -100
        self.weight_decay = 0.01

In [ ]:
import math
import torch.nn.functional as F
from torch.nn.attention.bias import causal_lower_right   # マスク作成で利用　PyTorch 2.5 以降?


class CausalSelfAttention(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        assert config.d_model % config.nhead == 0
        self.nhead = config.nhead
        self.head_dim = config.d_model // config.nhead

        # q,k,v をまとめて表現
        self.qkv = nn.Linear(config.d_model, 3 * config.d_model, bias=False)
        self.proj = nn.Linear(config.d_model, config.d_model, bias=False)
        self.attn_dropout_p = config.dropout
        self.resid_dropout = nn.Dropout(config.dropout)

    # 戻り値が２つになるので注意
    def forward(self, x, kv_cache=None, use_cache=False):
        bs, seq_len, d_model = x.shape
        q, k, v = self.qkv(x).split(d_model, dim=2)
        # (bs, seq_len, d_model) -> (bs, nhead, seq_len, head_dim)
        q = q.view(bs, seq_len, self.nhead, self.head_dim).transpose(1, 2)
        k = k.view(bs, seq_len, self.nhead, self.head_dim).transpose(1, 2)
        v = v.view(bs, seq_len, self.nhead, self.head_dim).transpose(1, 2)

        # 過去の k,v を連結（RoPEを入れる時はここより「前」で q,k を回転させる）
        # q は連結しないので、q_len は seq_len のまま
        if kv_cache is not None:                  # kv_cacheに値がある場合 (not None)
            past_k, past_v = kv_cache             # kv_cacheの値をpast_k, past_vへ代入
            k = torch.cat([past_k, k], dim=2)     # 新しいkとvを追記
            v = torch.cat([past_v, v], dim=2)
        new_cache = (k, v) if use_cache else None # use_cacheフラグがTrueのときは、追記したkとvをnew_cacheへ

        # 右下詰めの因果マスク。
        y = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=causal_lower_right(seq_len, k.size(2)),        # 学習時：(seq_len, seq_len)、生成時：(１, 過去の系列長)
            dropout_p=self.attn_dropout_p if self.training else 0.0,
        )

        y = y.transpose(1, 2).contiguous().view(bs, seq_len, d_model)
        y = self.resid_dropout(self.proj(y))
        return y, new_cache

class FFN(nn.Module):
    """SwiGLUの定義に従って書き下したバージョン"""
    def __init__(self, config):
        super().__init__()
        hidden = config.dim_feedforward
        self.w_gate = nn.Linear(config.d_model, hidden, bias=False)
        self.w_up   = nn.Linear(config.d_model, hidden, bias=False)
        self.w_down = nn.Linear(hidden, config.d_model, bias=False)
        self.activation = nn.SiLU()
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        gate = self.activation(self.w_gate(x))    # ゲート側
        up   = self.w_up(x)                       # そのまま通過の特徴量
        return self.dropout(self.w_down(gate*up)) # 線形層でMLPと同様に形状を整える出力


class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.attn_norm = nn.RMSNorm(config.d_model, eps=1e-6)
        self.attn = CausalSelfAttention(config)
        self.ffn_norm = nn.RMSNorm(config.d_model, eps=1e-6)
        self.ffn = FFN(config)

    def forward(self, x, kv_cache=None, use_cache=False):
        attn_out, new_cache = self.attn(self.attn_norm(x), kv_cache, use_cache)  # Attention(RMSNorm(x))
        x = x + attn_out                                                         # x + Attention(RMSNorm(x))
        x = x + self.ffn(self.ffn_norm(x))                                       # x + MLP(RMSNorm(x))
        return x, new_cache

In [ ]:
class DNN(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.config = config

        # 埋め込み層
        self.token_embedding = nn.Embedding(num_embeddings=config.vocab_size, embedding_dim=config.d_model)
        self.pos_embedding = nn.Embedding(num_embeddings=config.seq_len, embedding_dim=config.d_model)
        self.dropout = nn.Dropout(config.dropout)

        # causal transformer layer
        # SequentialからModuleListへ修正
        #self.transformer_blocks = nn.Sequential(*[TransformerBlock(config) for _ in range(config.num_layers)])
        self.transformer_blocks = nn.ModuleList([TransformerBlock(config) for _ in range(config.num_layers)])

        # 最後の出力に向けた正規化とFC層　最終的に単語数になる
        self.norm = nn.RMSNorm(config.d_model, eps=1e-6)
        self.fc = nn.Linear(config.d_model, config.vocab_size,bias=False)
        self.apply(self._init_weights)                    # 個別に初期化
        self.fc.weight = self.token_embedding.weight      # 重み共有


        # 残差方向の出力層だけ 1/sqrt(2N) スケール
        std = 0.02 / math.sqrt(2 * config.num_layers)
        for pn, p in self.named_parameters():
            if pn.endswith("proj.weight") or pn.endswith("w_down.weight"):
                nn.init.normal_(p, mean=0.0, std=std)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:          # 全層bias=False なのでスキップ
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.padding_idx is not None:   #  padding_idxは未指定なのでスキップ
                with torch.no_grad():
                    module.weight[module.padding_idx].zero_()


    def forward(self, x, kv_caches=None, use_cache=False):
        bs, seq_len = x.shape

        if kv_caches is None:    # kv_cashesなし：学習時・初回
            past_len = 0
        else:                    # kv_cashesある時：生成２回目以降
            past_len = kv_caches[0][0].size(2)
        assert past_len + seq_len <= self.config.seq_len, "位置埋め込みの範囲外 seq_lenを超えています"

        positions = torch.arange(past_len, past_len + seq_len, device=x.device)
        tok_emb = self.token_embedding(x)
        pos_emb = self.pos_embedding(positions).unsqueeze(0)
        x = tok_emb + pos_emb           # トークン埋め込み＋位置埋め込み
        x = self.dropout(x)

        new_caches = [] if use_cache else None
        for i, block in enumerate(self.transformer_blocks):
            cache = None if kv_caches is None else kv_caches[i]
            x, new_cache = block(x, kv_cache=cache, use_cache=use_cache)
            if use_cache:
                new_caches.append(new_cache)
        # use_cache=True and seq_len>1なので基本Prefill時に実行される
        # x[:,-1:]によって (batch_size=1, seq_len, d_model) -> (batch_size=1, 1, d_model) へ
        if use_cache and seq_len > 1:
            x = x[:, -1:]

        logits = self.fc(self.norm(x))
        return logits, new_caches


In [ ]:
config = ModelConfig(tokenizer)
model = DNN(config).to(device)

In [ ]:
from torchinfo import summary

summary(
    model,
    input_size=(1, config.seq_len),        # (batch, seq_len)
    dtypes=[torch.long],                   # 埋め込みの入力は整数インデックス
    #col_names=("input_size", "output_size", "num_params"),
    depth=2,
)

In [ ]:
unique = sum(p.numel() for p in model.parameters())
total  = sum(p.numel() for _, p in model.named_parameters(remove_duplicate=False))
print(f"unique={unique:,}  reported={total:,}  tied={total-unique:,}")

### Random化

In [ ]:
class RandomGPTDataset(Dataset):
    def __init__(
        self,
        data_dir,
        tokenizer,
        context_size=128,
        samples_per_epoch=10_000,
    ):
        self.context_size = context_size
        self.samples_per_epoch = samples_per_epoch
        eod_id = tokenizer.token_to_id("<eod>")
        filenames = sorted(Path(data_dir).glob("*.txt"))

        all_ids = []
        for filename in filenames:
            text = filename.read_text(encoding="utf-8")
            document_ids = tokenizer.encode(text, add_special_tokens=False).ids
            all_ids.extend(document_ids)
            all_ids.append(eod_id)

        self.ids = torch.tensor(all_ids, dtype=torch.long)

    def __len__(self):
        # 1 epochあたりに何個のランダム窓を学習するか
        # 10,000個（回？）がデフォルト値
        return self.samples_per_epoch

    def __getitem__(self, idx):
        # idxは使わず、毎回ランダムな開始位置を選ぶ
        start = torch.randint(
            low=0,
            high=len(self.ids) - self.context_size,
            size=(1,),
        ).item()

        x = self.ids[start:start + self.context_size]
        y = self.ids[start + 1:start + self.context_size + 1]

        return {"ids": x, "labels": y}

In [ ]:
dataset = RandomGPTDataset(
    data_dir=data_dir,
    tokenizer=tokenizer,
    context_size=config.context_size,
    samples_per_epoch=10_000,
)

dataloader = DataLoader(
    dataset=dataset,
    batch_size=config.batch_size,  # メモリ足りない場合は小さくする
    shuffle=False,
    drop_last=True,
    num_workers=0, # 2,4のほうが早い？要検討
    pin_memory=True,
)


In [ ]:
decay, no_decay = [], []
for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    if param.dim() >= 2:
        decay.append(param)      # Linear の重み、Embedding
    else:
        no_decay.append(param)   # RMSNorm の gain、bias

optim_groups = [
    {"params": decay,    "weight_decay": config.weight_decay},
    {"params": no_decay, "weight_decay": 0.0},
]
optimizer = torch.optim.AdamW(optim_groups, lr=config.learning_rate, betas=(0.9, 0.95), fused=(device.type == "cuda"))

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=config.ignore_index)    # 今回ignore_indexを利用していないがSFT用につけておく

In [ ]:
# step数で管理するためのデータローダー関数
def infinite_loader(dataloader):
    while True:
        for batch in dataloader:
            yield batch

data_iter = infinite_loader(dataloader)  # epochではなく、step数で計測

In [ ]:
from tqdm import tqdm
max_iters = config.num_steps  # step数
total_token = 0               # 累積トークン数を初期化
pbar = tqdm(range(max_iters))

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()


model.train()                 # trainモードを明示
for step in pbar:
    batch = next(data_iter)
    input_ids = batch["ids"].to(device, non_blocking=True)
    labels = batch["labels"].to(device, non_blocking=True)
    optimizer.zero_grad()
    # (1) GPUが対応している場合は、bf16へ変更してflash attentionを使う
    with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=use_bf16):
        outputs, _ = model(input_ids)
        loss = criterion(
            outputs.view(-1, config.vocab_size),
            labels.view(-1),
        )
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=config.max_grad_norm)  # 勾配クリップ
    optimizer.step()

    # このstepのトークン数 （バッチサイズ×系列長）を加算
    total_token += input_ids.shape[0] * input_ids.shape[1]

    # set_postfixは辞書が引数になる
    pbar.set_postfix({"loss": f"{loss.item():.4f}", "tokens": f"{total_token/1e6:.2f}M"})
    if (step+1) % 500 == 0:
        tqdm.write(f"{step+1}-step:\tloss:{loss.item():.3f}\ttokens:{total_token:,}")

### 文章生成

In [ ]:
@torch.inference_mode()
def generate_text(model, input_ids, max_new_tokens=100, eos_token_id=None):
    model.eval()
    generated = input_ids.to(device)
    max_len = model.config.seq_len

    # プリフィル：プロンプト全体を1回だけ流す
    logits, caches = model(generated, use_cache=True)
    next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
    generated = torch.cat([generated, next_token], dim=1)

    for _ in range(max_new_tokens - 1):
        if eos_token_id is not None and next_token.item() == eos_token_id:
            break
        if generated.size(1) >= max_len:
            break
        # デコード：直前の1トークンだけを流す
        logits, caches = model(next_token, kv_caches=caches, use_cache=True)
        next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
        generated = torch.cat([generated, next_token], dim=1)

    return generated

In [ ]:
from tokenizers import decoders
import pandas as pd
tokenizer.decoder = decoders.ByteLevel() # BPEのときはこれが必要、文字化けするぞ

def response(prompt):
    model.eval()
    model_input = tokenizer.encode(prompt, add_special_tokens=False)
    input_ids = torch.tensor([model_input.ids], dtype=torch.long, device=device)

    output_ids = generate_text(
        model=model,
        input_ids=input_ids,
        max_new_tokens=config.seq_len,
        eos_token_id=config.eod_token_id,
    )

    # 全文をデコード
    generated_text = tokenizer.decode(output_ids[0].tolist(), skip_special_tokens=False)
    print(generated_text)

df = pd.read_csv("./data/homme_prompt.csv")

import time
start = time.perf_counter()
for num, prompt in enumerate(df["prompt"], 1):
    print(f"【{num}番目】")
    response(prompt)
end = time.perf_counter()
print(f"ループ全体の実行時間: {end - start :.3f} 秒")

In [ ]:
torch.save({
        "model_state_dict": model.state_dict(),
        "config": config.__dict__,  # configも一緒に保存
        }, pretrain_filename)

In [ ]:
checkpoint = torch.load(pretrain_filename)
config = ModelConfig(tokenizer)
config.__dict__.update(checkpoint["config"])
model = DNN(config).to(device)
model.load_state_dict(checkpoint["model_state_dict"])